<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/premi-secundaria/++theme++genweb6.theme/img/logo.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 10px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Model Risk & Data Science Training</b></p>
      <p style="margin: 0;">Automatización y robotización de procesos</p>
  </div>

</div>

# Bloque 09 · Automatización de PDF

Caso del Observatorio documental. Ejecuta las celdas en orden desde la carpeta del bloque. Lee la explicación, modifica un dato del ejemplo y comprueba cómo cambia el resultado. Los archivos generados quedan en una carpeta temporal.

Instala las dependencias con `%pip install -r requirements.txt`. La demostración de OCR requiere Tesseract en el PATH y utiliza una imagen incluida.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

temporal = TemporaryDirectory()
salida = Path(temporal.name)
ejemplos = Path("ejemplos")
import json, re, shutil, subprocess
from collections import Counter
from pypdf import PdfReader, PdfWriter
from pdf_observatorio import inspeccionar, inventario, seleccionar

## Ejemplo 1 · Páginas, objetos y geometría

Compara un PDF digital, uno escaneado y uno mixto página a página. MediaBox describe el tamaño físico y CropBox el área visible. Una página sin texto extraíble puede contener una imagen con texto legible.

In [ ]:
for ruta_pdf in sorted(ejemplos.glob("*.pdf")):
    for numero, pagina in enumerate(PdfReader(ruta_pdf).pages, 1):
        texto = (pagina.extract_text() or "").strip()
        print(ruta_pdf.name, numero, tuple(pagina.mediabox), tuple(pagina.cropbox),
              "rotación:", pagina.rotation, "caracteres:", len(texto), "imágenes:", len(pagina.images))

## Ejemplo 2 · Orden visual y extracción

La extracción básica sigue las instrucciones del PDF. El modo layout intenta conservar la disposición mediante espacios. Para una tabla regular, ensaya la separación de columnas por espaciado y revisa cabecera y número de campos antes de tratar las filas como datos.

In [ ]:
digital = ejemplos / "01_digital.pdf"
mixto = ejemplos / "03_mixto.pdf"
pagina_digital = PdfReader(digital).pages[0]
print(pagina_digital.extract_text())
print(pagina_digital.extract_text(extraction_mode="layout"))
layout_tabla = "Código     Estado\nDOC-001    Revisado\nDOC-002    Pendiente"
filas_tabla = [re.split(r" {2,}", linea.strip()) for linea in layout_tabla.splitlines()]
assert all(len(fila) == 2 for fila in filas_tabla)
print(filas_tabla)

## Ejemplo 3 · Patrones, ambigüedad y procedencia

Buscamos identificadores con tres dígitos ASCII y límites de palabra. Distinguimos ausencia, identificador único y varios candidatos. Conservamos la página para volver al original.

In [ ]:
patron = r"\bDOC-[0-9]{3}\b"
for nombre, numero, texto in [(digital.name, 1, pagina_digital.extract_text()),
                             ("simulado.pdf", 2, "DOC-009 DOC-010 DOC-010 DOC-1234")]:
    candidatos = sorted(set(re.findall(patron, texto or "")))
    estado = {0: "ausente", 1: "unico"}.get(len(candidatos), "ambiguo")
    print(nombre, numero, estado, candidatos)

## Ejemplo 4 · OCR y revisión

Extraemos la imagen de la página escaneada y ejecutamos Tesseract si está disponible. El OCR puede confundir O con 0 y un código plausible requiere contraste con la imagen o un catálogo. Conservamos el estado original de extracción al crear el índice.

In [ ]:
escaneado = ejemplos / "02_escaneado.pdf"
imagen = PdfReader(escaneado).pages[0].images[0]
ruta_imagen = salida / Path(imagen.name).name
ruta_imagen.write_bytes(imagen.data)
motor = shutil.which("tesseract")
if motor:
    ocr = subprocess.run([motor, str(ruta_imagen), "stdout"], capture_output=True,
                         text=True, encoding="utf-8", errors="replace", timeout=30)
    print(ocr.stdout.strip() if ocr.returncode == 0 else ocr.stderr.strip())
else:
    print("Tesseract no disponible. Caso simulado de lectura dudosa: DOC-0O2")

## Ejemplo 5 · Selección, combinación y división

La interfaz numera desde 1 y pypdf indexa desde 0. Una selección ordenada conserva el orden pedido. Para unir documentos, se añaden en el orden de entrega. Para dividir, se crean archivos con rangos explícitos.

In [ ]:
invertido = salida / "invertido.pdf"
invertido.unlink(missing_ok=True)
seleccionar(mixto, invertido, [2, 1])
assert not (PdfReader(invertido).pages[0].extract_text() or "").strip()
combinador = PdfWriter()
for ruta_pdf in (digital, mixto):
    combinador.append(ruta_pdf)
unido = salida / "unido.pdf"
combinador.write(unido)
for i, rango in enumerate(([1], [2, 3]), 1):
    parte = salida / f"parte_{i}.pdf"
    parte.unlink(missing_ok=True)
    seleccionar(unido, parte, rango)
    assert len(PdfReader(parte).pages) == len(rango)

## Ejemplo 6 · Rotación, recorte y comprobación

Trabajamos en una copia. La rotación altera la vista de la página. El recorte cambia la región visible y el contenido oculto puede seguir siendo extraíble. Una ocultación visual no sirve para eliminar información.

In [ ]:
editor = PdfWriter()
copia = editor.add_page(pagina_digital)
copia.rotate(90)
copia.cropbox.upper_right = (float(copia.mediabox.right) / 2, float(copia.mediabox.top))
vista = salida / "vista.pdf"
editor.write(vista)
verificada = PdfReader(vista).pages[0]
assert verificada.rotation == 90
print(verificada.cropbox, re.findall(patron, verificada.extract_text() or ""))

## Ejemplo 7 · Lote con estados y coste

Primero extrae texto codificado. Deja OCR para las páginas candidatas y registra los fallos por archivo. La verificación compara estructura, contenido y procedencia con el índice de referencia.

In [ ]:
indice = inventario(ejemplos)
esperado = json.loads((ejemplos / "indice_esperado.json").read_text(encoding="utf-8"))
assert indice == esperado
pendientes_pdf = [(d["archivo"], p["pagina"]) for d in indice for p in d["paginas"]
                  if p["estado"] == "sin_texto_extraido"]
print("Cola de revisión/OCR:", pendientes_pdf)
(salida / "indice.json").write_text(json.dumps(indice, indent=2, ensure_ascii=False), encoding="utf-8")

## Diez ejercicios

Completa las celdas TODO. Cada enunciado incluye una comprobación para validar tu resultado.

### Ejercicio 01 · Medir una página

Guarda en medidas la anchura y altura de MediaBox de la primera página digital.

In [ ]:
# TODO · escribe aquí tu solución

# Cuando termines, descomenta la comprobación.
# assert len(medidas) == 2 and min(medidas) > 0

### Ejercicio 02 · Clasificar un PDF mixto

Recorre las páginas del mixto y guarda un booleano por página en tiene_texto.

In [ ]:
# TODO · escribe aquí tu solución

# Cuando termines, descomenta la comprobación.
# assert tiene_texto == [True, False]

### Ejercicio 03 · Validar códigos candidatos

Extrae códigos distintos de «DOC-015 DOC-015 DOC-020 DOC-1234». Guarda la lista ordenada en ids.

In [ ]:
# TODO · escribe aquí tu solución

# Cuando termines, descomenta la comprobación.
# assert ids == ["DOC-015", "DOC-020"]

### Ejercicio 04 · Revisar una salida OCR

Clasifica como válidos en candidatos_ocr los tokens completos de «DOC-001 DOC-0O2 DOC-003». Conserva el sospechoso en revisar_ocr para contrastarlo con la imagen.

In [ ]:
# TODO · escribe aquí tu solución

# Cuando termines, descomenta la comprobación.
# assert candidatos_ocr == ["DOC-001", "DOC-003"] and revisar_ocr == ["DOC-0O2"]

### Ejercicio 05 · Reconstruir una tabla

Convierte filas_tabla en una lista de diccionarios con las claves de la primera fila. Guarda en registros.

In [ ]:
# TODO · escribe aquí tu solución

# Cuando termines, descomenta la comprobación.
# assert registros[1] == {"Código": "DOC-002", "Estado": "Pendiente"}

### Ejercicio 06 · Extraer un rango

Selecciona la página 2 de mixto en rango_pdf. Comprueba número de páginas y ausencia de texto.

In [ ]:
# TODO · escribe aquí tu solución

# Cuando termines, descomenta la comprobación.
# assert len(PdfReader(rango_pdf).pages) == 1 and not PdfReader(rango_pdf).pages[0].extract_text().strip()

### Ejercicio 07 · Combinar en orden explícito

Une mixto y digital, en ese orden, en orden_pdf.

In [ ]:
# TODO · escribe aquí tu solución

# Cuando termines, descomenta la comprobación.
# assert len(PdfReader(orden_pdf).pages) == 3 and "DOC-001" in PdfReader(orden_pdf).pages[-1].extract_text()

### Ejercicio 08 · Rotar sin tocar el original

Escribe la primera página de digital rotada 180° en rotado.

In [ ]:
# TODO · escribe aquí tu solución

# Cuando termines, descomenta la comprobación.
# assert PdfReader(rotado).pages[0].rotation == 180 and pagina_digital.rotation == 0

### Ejercicio 09 · Aislar errores del lote

Crea una carpeta temporal con un PDF cifrado y otro corrupto. Ejecuta inventario y cuenta estados en estados_lote.

In [ ]:
# TODO · escribe aquí tu solución

# Cuando termines, descomenta la comprobación.
# assert estados_lote == {"cifrado": 1, "error_lectura": 1}

### Ejercicio 10 · Construir una cola auditable

Desde el índice, crea cola como lista de tuplas (archivo, página) para revisión. Comprueba que cubre dos archivos.

In [ ]:
# TODO · escribe aquí tu solución

# Cuando termines, descomenta la comprobación.
# assert cola == [("02_escaneado.pdf", 1), ("03_mixto.pdf", 2)]